In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

define plot types

In [ ]:
import sys
sys.path.append("../utils")
from plotting_utils import format_top_3, plot_metric_grouped_by

In [ ]:
base_path = ".."
mali_path = "../../MALI"
scratch_path = ".."
batch = 4




create a new summary df based on aggregating the dfs in the n_comp subfolders

In [ ]:
# import os
# noises = [0.0, 0.5, 0.8, 1.0, 10.0]
# dropouts = [0.0, 0.1, 0.4, 0.8, 0.9]
# n_components_list = [2, 5, 10, 30]


# all_results_df = pd.DataFrame()
# for noise_std in noises:
#     for dropout_prob in dropouts:
#         for n_components in n_components_list:
#             save_path_subfolder = f"{save_path}/noise_{noise_std}_dropout_{dropout_prob}/{n_components}_components"

#             df = pd.read_csv(f"{save_path_subfolder}/simulated_batches_results.csv")

#             all_results_df = pd.concat([all_results_df, df], ignore_index=True)
        
# all_results_df    
# all_results_df.to_csv(f"{save_path}/simulated_batches_results.csv", index=False)
            

# Fake batches data

In [ ]:
results_name= "simulated_batches"
exp =f"{results_name}/9119203"
# exp = f"{results_name}_1000_trees"



batch = 4
results_path = f"{scratch_path}/results/{exp}/{batch}"
save_path = f"{scratch_path}/results/{exp}/{batch}"

# load results
results_df = pd.read_csv(f"{results_path}/{results_name}_results.csv")


metric_type = results_df.loc[results_df["method"] == "Metric Type"]
metric_type = metric_type.drop(columns=['method', "noise_std", "dropout_prob"]).drop_duplicates()
results_df = results_df.loc[results_df["method"] != "Metric Type"]

# # convert relevant columns to numeric
num_cols = results_df.columns.difference(['method', 'noise_std', 'dropout_prob'])
results_df[num_cols] = results_df[num_cols].apply(pd.to_numeric, errors='coerce')

noise_dropout_cols = ['noise_std', 'dropout_prob']
results_df[noise_dropout_cols] = results_df[noise_dropout_cols].apply(pd.to_numeric, errors='coerce')


# results_df = results_df.drop(0, axis = 0) # remove one run from debugging
results_df.drop_duplicates(subset=['n_components','method', 'noise_std', 'dropout_prob'], keep='last', inplace=True)


# delete rows where n_components is NA
results_df = results_df.loc[~results_df["n_components"].isna()]
results_df = results_df.loc[~results_df["method"].isna()]


# remove n_components in the model column (it was just used to not overwrite adata, and it's already another column)
results_df["method"] = results_df["method"].str.replace(r'_\d+_components', '', regex=True)

In [ ]:
results_df["method"].unique()

remove noise_std = 10 because that's really high and unrealistic (negative values in the counts?)

In [ ]:
results_df = results_df.loc[results_df["noise_std"] != 10]
results_df = results_df.loc[results_df["noise_std"] != 2]

remove some of our methods

In [ ]:
methods_to_remove= ["FoSTA_spectral", "FoSTA_UMAP", "RFMALI_spectral", "RFMALI_UMAP", "FoSTA_PHATE_t1", "RFMALI_PHATE_t1"]
# methods_to_remove += ["RFMALI_PHATE_t2"]
results_df = results_df[~results_df["method"].isin(methods_to_remove)]

# rename FOSTA PHATE t2 to FoSTA and RFMALI PHATE t2 to RFMALI
results_df["method"] = results_df["method"].replace({"FoSTA_PHATE_t2": "FoSTA"})
results_df["method"] = results_df["method"].replace({"RFMALI_PHATE_t2": "RFMALI"})

reorder the methods

In [ ]:
methods_order = results_df["method"].unique().tolist()

for method in ["RFMALI", "FoSTA"]:
    if method in methods_order:
        methods_order.remove(method)
        methods_order.insert(0, method)

for method in ["Unintegrated"]:
    methods_order.remove(method)
    methods_order.append(method)

print(methods_order)

results_df["method"] = pd.Categorical(results_df["method"], ordered=True, categories = methods_order)

In [ ]:
results_df.loc[results_df["Total"].isna()] # there are no NAs. everything ran?

In [ ]:
results_df

restrict to 2 components

In [ ]:
results_df = results_df.loc[results_df["n_components"] == 2]

In [ ]:
# cols_to_plot = ["Batch correction", "Bio conservation", "Total"]
# # 
# for col in cols_to_plot:
#     fig, ax = plt.subplots(figsize=(18,7), layout='constrained')


#     data = results_df.groupby(['n_components', 'method'])[col].mean()
#     # print(data)
   
#     sns.barplot(data=results_df, x = "n_components", y=col, hue="method", palette ="tab20")
#         # move legend outside   
    
    
#     plt.title(f"{col} by n_components")
#     plt.xticks(rotation=90)
#     plt.show()

# tables of bio and batch metrics

In [ ]:
bio_metrics = []
batch_metrics = []
for metric in metric_type.columns:
    if (metric_type[metric] == "Bio conservation").values[0]:
        bio_metrics.append(metric)
        
    elif (metric_type[metric] == "Batch correction").values[0]:
        batch_metrics.append(metric)

In [ ]:
bio_avg = results_df.groupby(['method'])[bio_metrics].mean()
bio_std = results_df.groupby(['method'])[bio_metrics].std()

In [ ]:
batch_avg = results_df.groupby(['method'])[batch_metrics].mean()
batch_std = results_df.groupby(['method'])[batch_metrics].std()

In [ ]:
# change all columns to str
bio_avg = bio_avg.astype(float)
batch_avg = batch_avg.astype(float)

In [ ]:
bio_avg

In [ ]:
format_top_3(bio_avg)

In [ ]:

# add std
def add_std(summary_df, summary_df_std):
    summary_df_fmt = summary_df.copy()
    for col in summary_df.columns:
        summary_df_fmt[col] = (
            summary_df[col].astype(str)
            + r" {\small $\pm$ "
            + summary_df_std[col].round(3).astype(str)
            + "}"
        )
    return summary_df_fmt

bio = add_std(format_top_3(bio_avg), bio_std)
batch = add_std(format_top_3(batch_avg), batch_std)

In [ ]:
import warnings
with warnings.catch_warnings():
    warnings.simplefilter("ignore")
    print(batch.to_latex(
        escape=False,
        multirow=True,
        float_format="%.3f"
    )) 

# then copy the output latex table into a .tex file for inclusion in the paper

In [ ]:
import warnings
with warnings.catch_warnings():
    warnings.simplefilter("ignore")
    print(bio.to_latex(
        escape=False,
        multirow=True,
        float_format="%.3f"
    )) 

# then copy the output latex table into a .tex file for inclusion in the paper

In [ ]:
import warnings
with warnings.catch_warnings():
    warnings.simplefilter("ignore")
    print(format_top_3(bio_avg).to_latex(
        escape=False,
        multirow=True,
        float_format="%.3f"
    )) 

# then copy the output latex table into a .tex file for inclusion in the paper

# box plot

In [ ]:
def boxplot(results_df, col="Total", savedir = None):
    
    fig, ax = plt.subplots()
    
    order_desc = results_df.groupby("method")[col].mean().sort_values(ascending=False).index.tolist()
    
    sns.barplot(data=results_df, x="method", y=col, palette="tab20", ax=ax, order = order_desc)
    plt.title(f"{col} score by method")

    # ax.set_title(f"{col}")
    ax.tick_params(axis="x", rotation=90)
    
    plt.tight_layout()

    if savedir is not None:
        plt.savefig(f"{savedir}/{col}.pdf", format='pdf')
        plt.savefig(f"{savedir}/{col}.png", format='png')

In [ ]:
boxplot(results_df, col="Bio conservation", savedir = save_path)

# Bar plots

In [ ]:
plot_metric_grouped_by(results_df, groupby_cols=["n_components", "method"])

subset to 2 components for visualization

In [ ]:
plot_metric_grouped_by(results_df, groupby_cols=["n_components","method"], n_components=2)

effect of noise when fixing dropout at 0.1:

In [ ]:
# pick one n_components and see effect of noise and dropout

n_comp = 2.0
dropout = 0.0

results_df_subset = results_df.loc[(results_df["n_components"] == n_comp) & (results_df["dropout_prob"] == dropout)]

plot_metric_grouped_by(results_df_subset, groupby_cols=["noise_std", "method"])


effect of dropout when fixing noise at 0

In [ ]:
# pick one n_components and see effect of noise and dropout

n_comp = 2
noise = 0

results_df_subset = results_df.loc[(results_df["n_components"] == n_comp) & (results_df["noise_std"] == noise)]

plot_metric_grouped_by(results_df_subset, groupby_cols=["dropout_prob", "method"])

In [ ]:
# # average over all components....
# cols_to_plot = ["Batch correction", "Bio conservation", "Total"]
# # 
# for col in cols_to_plot:
#    fig, ax = plt.subplots(figsize=(8, 6))
   
   
#    data = results_df.groupby(['n_components', 'method'])[col].mean()
#    print(data)
   
# #    data.plot(kind='bar', title=col)
#    for n_comp in results_df['n_components'].unique():
#       sns.barplot(data=data[n_comp])
   
   
#       # show the mean
#       for p in ax.patches:
#          h, w, x = p.get_height(), p.get_width(), p.get_x()
#          xy = (x + w / 2., h / 2)
#          text = f'{h:0.3f}'
#          ax.annotate(text=text, xy=xy, ha='center', va='top')
         
#       plt.title(f"{col} - n_components: {n_comp}")
#       plt.xticks(rotation=90)
#       plt.show()

for each method, heatmap of performance over different noise/dropout

In [ ]:
# for method in results_df['method'].unique():
#     method_df = results_df[results_df['method'] == method]
#     # results_df["Total"] = pd.to_numeric(results_df["Total"], errors="coerce")
#     heatmap_df = method_df.pivot(index= "dropout_prob", columns= "noise_std", values = "Total")
    
#     # method_df = method_df.mean().to_frame(name='score')
    
#     plt.figure(figsize=(10,6))
#     sns.heatmap(data=heatmap_df, vmin=0, vmax=1, cmap= "viridis")
#     plt.title(f"Total Score for Method: {method}")
#     plt.ylabel("Dropout Probability")
#     plt.xlabel("Noise Standard Deviation")
    

# scatter plot of bio vs batch

In [ ]:
n_components = 2
results_df_n_comp = results_df[results_df["n_components"] == n_components]

results_df_subset = results_df_n_comp
# # if i fix no noise, only dropout:
# results_df_subset = results_df_n_comp[results_df_n_comp["noise_std"] == 10]

# results_df_subset = results_df_n_comp[results_df_n_comp["noise_std"] == 1.0]
# if i fix no noise, only dropout:
# results_df_subset = results_df_n_comp[results_df_n_comp["dropout_prob"] == 0.8]



plot_bio_vs_batch_correction(results_df_subset, save_path=save_path)


In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# ... (Previous code for filtering results_df_subset) ...
n_components = 2
results_df_n_comp = results_df[results_df["n_components"] == n_components]
results_df_subset = results_df_n_comp

def plot_bio_vs_batch_correction(results_df, save_path=None):
    # 1. Calculate Mean and Standard Deviation
    mean_df = results_df.groupby(['method'])[["Bio conservation", "Batch correction"]].mean()
    std_df = results_df.groupby(['method'])[["Bio conservation", "Batch correction"]].std()
    std_df = std_df.fillna(0)

    # 2. Calculate Ranks (Assuming higher score is better -> ascending=False)
    mean_df['Bio Rank'] = mean_df['Bio conservation'].rank(ascending=False).astype(int)
    mean_df['Batch Rank'] = mean_df['Batch correction'].rank(ascending=False).astype(int)
    
    # 3. Calculate Average Rank and Sort
    mean_df['Avg Rank'] = (mean_df['Bio Rank'] + mean_df['Batch Rank']) / 2
    mean_df = mean_df.sort_values('Avg Rank')

    plt.figure(figsize=(10, 8)) 
    
    # Use a fixed color map so colors stay consistent regardless of sorting
    unique_methods = results_df['method'].unique()
    # Create palette
    palette_list = sns.color_palette("tab20", n_colors=len(unique_methods))
    color_map = dict(zip(unique_methods, palette_list))
    
    # Iterate through the SORTED methods
    for method in mean_df.index:
        x = mean_df.loc[method, "Batch correction"]
        y = mean_df.loc[method, "Bio conservation"]
        x_err = std_df.loc[method, "Batch correction"]
        y_err = std_df.loc[method, "Bio conservation"]
        
        bio_rank = mean_df.loc[method, 'Bio Rank']
        batch_rank = mean_df.loc[method, 'Batch Rank']
        avg_rank = mean_df.loc[method, 'Avg Rank']

        # Legend Label: Includes Bio and Batch ranks
        legend_label = f"{method} (Bio #{bio_rank}, Batch #{batch_rank})"

        # 4. Plot Error Bars
        plt.errorbar(
            x, y, 
            xerr=x_err, 
            yerr=y_err, 
            fmt='none',
            ecolor=color_map[method],
            elinewidth=1.5,
            capsize=5,
            alpha=0.4, 
            label=None 
        )

        # 5. Plot the Marker (No annotation on plot)
        plt.scatter(
            x, y, 
            s=100, 
            color=color_map[method], 
            label=legend_label, 
            zorder=3
        )

    # 6. Final Formatting
    plt.legend(
        bbox_to_anchor=(1.05, 1), 
        loc='upper left', 
        title="Method (Ordered by Avg Rank)",
        frameon=True
    )
    
    plt.title("Bio conservation vs Batch correction")
    plt.xlabel("Batch correction")
    plt.ylabel("Bio conservation")
    plt.grid(True, linestyle='--', alpha=0.5)

    if save_path is not None:
        plt.tight_layout()
        plt.savefig(f"{save_path}/bio_vs_batch_correction_ncomp_{n_components}.pdf", format='pdf')
        plt.savefig(f"{save_path}/bio_vs_batch_correction_ncomp_{n_components}.png", format='png')
    plt.show()

plot_bio_vs_batch_correction(results_df_subset, save_path=save_path)

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# ... (Previous code for filtering results_df_subset) ...
n_components = 2
results_df_n_comp = results_df[results_df["n_components"] == n_components]
results_df_subset = results_df_n_comp

def plot_bio_vs_batch_correction(results_df, save_path=None):
    # 1. Calculate Mean and Standard Deviation
    mean_df = results_df.groupby(['method'])[["Bio conservation", "Batch correction"]].mean()
    std_df = results_df.groupby(['method'])[["Bio conservation", "Batch correction"]].std()
    std_df = std_df.fillna(0)

    # 2. Calculate Ranks
    mean_df['Bio Rank'] = mean_df['Bio conservation'].rank(ascending=False).astype(int)
    mean_df['Batch Rank'] = mean_df['Batch correction'].rank(ascending=False).astype(int)
    
    # 3. Calculate Average Rank and Sort
    mean_df['Avg Rank'] = (mean_df['Bio Rank'] + mean_df['Batch Rank']) / 2
    mean_df = mean_df.sort_values('Avg Rank')

    # --- ADJUSTMENT: Smaller Figure Size ---
    plt.figure(figsize=(5, 5)) 
    
    unique_methods = results_df['method'].unique()
    palette_list = sns.color_palette("tab20", n_colors=len(unique_methods))
    color_map = dict(zip(unique_methods, palette_list))
    
    for method in mean_df.index:
        x = mean_df.loc[method, "Batch correction"]
        y = mean_df.loc[method, "Bio conservation"]
        x_err = std_df.loc[method, "Batch correction"]
        y_err = std_df.loc[method, "Bio conservation"]
        
        bio_rank = mean_df.loc[method, 'Bio Rank']
        batch_rank = mean_df.loc[method, 'Batch Rank']
        legend_label = f"{method} (Bio #{bio_rank}, Batch #{batch_rank})"

        # Plot Error Bars
        plt.errorbar(
            x, y, 
            xerr=x_err, 
            yerr=y_err, 
            fmt='none',
            ecolor=color_map[method],
            elinewidth=1.5,
            capsize=5,
            alpha=0.4, 
            label=None 
        )

        # Plot Marker
        plt.scatter(
            x, y, 
            s=100, 
            color=color_map[method], 
            label=legend_label, 
            zorder=3
        )

    # --- ADJUSTMENT: Legend Styling ---
    plt.legend(
        bbox_to_anchor=(1.05, 1), 
        loc='upper left', 
        title="Method (Ordered by Avg Rank)",
        frameon=True,
        fontsize='small',  # Make text smaller to fit the smaller figure height
        title_fontsize='medium'
    )
    
    plt.title("Bio conservation vs Batch correction")
    plt.xlabel("Batch correction")
    plt.ylabel("Bio conservation")
    plt.grid(True, linestyle='--', alpha=0.5)

    if save_path is not None:
        # 'bbox_inches="tight"' ensures the external legend is not cut off when saving
        plt.savefig(f"{save_path}/bio_vs_batch_correction_ncomp_{n_components}.pdf", format='pdf', bbox_inches="tight")
        plt.savefig(f"{save_path}/bio_vs_batch_correction_ncomp_{n_components}.png", format='png', bbox_inches="tight")
    plt.show()

plot_bio_vs_batch_correction(results_df_subset, save_path=save_path)

all points for all methods

In [ ]:
data = results_df_n_comp.groupby(['noise_std', 'dropout_prob', 'method'])[["Bio conservation", "Batch correction"]].mean()
    
sns.scatterplot(data=data, x="Batch correction", y="Bio conservation", hue = "method", palette="tab20")

In [ ]:
# for method in results_df["method"].unique():
#     method_df = results_df[results_df["method"] == method]
#     data = method_df.groupby(['noise_std', 'dropout_prob'])[["Bio conservation", "Batch correction"]].mean()
    
#     sns.scatterplot(data=data, x="Batch correction", y="Bio conservation")
#     plt.title(f"Method: {method}")
#     plt.show()

# clean up cell types

# visualize embeddings

In [ ]:
import scanpy as sc
import matplotlib.pyplot as plt


def load_adata(noise_level=0.5, dropout_level=0.5):
    # format noise and dropout to match folder names (1 decimal place)
    noise_level = f"{noise_level:.1f}"
    dropout_level = f"{dropout_level:.1f}"
    path = f"{save_path}/noise_{noise_level}_dropout_{dropout_level}/2_components"
    adata = sc.read_h5ad(f"{path}/adata_intermediate.h5ad")
    # rename FoSTA_PHATE_t2 to FoSTA in the adata
    adata.obsm["FoSTA"] = adata.obsm["FoSTA_PHATE_t2"]
    del adata.obsm["FoSTA_PHATE_t2"]
    
    return adata, path


def plot_embeddings(adata = None, methods = None, noise_level=0.5, dropout_level=0.5, save_path = None, rename_dict = None, figsize=None, **kwargs):
    label_key = "cell_type"
    batch_key = "simulated_batch"
    

    if adata is None:
        adata, save_path = load_adata(noise_level=noise_level, dropout_level=dropout_level)
     
    if methods is None:
        methods = list(adata.obsm.keys())
    n = len(methods)
    
    if figsize is None:
        figsize = (6*n, 8)
        
    fig, axes = plt.subplots(figsize=figsize, nrows=2, ncols=n)


    # ---- TOP ROW (batch_key) ----
    for i, method in enumerate(methods):
        if rename_dict is None:
            method_name = method
        else:
            method_name = rename_dict[method]
            
        sc.pl.embedding(
            adata,
            ax=axes[0, i],
            basis=method,
            color=batch_key,
            title=f"{method_name}", # : {batch_key}
            show=False,
            legend_loc="right" if i == 0 else None,  # create legend ONCE
            **kwargs
        )

    # ---- BOTTOM ROW (label_key) ----
    for i, method in enumerate(methods):
        if rename_dict is None:
            method_name = method
        else:
            method_name = rename_dict[method]
            
        sc.pl.embedding(
            adata,
            ax=axes[1, i],
            basis=method,
            color=label_key,
            title=f"{method_name}", # {label_key}
            show=False,
            legend_loc="right" if i == 0 else None,
            **kwargs
        )

    # ---- EXTRACT + MOVE LEGENDS ----
    leg_top = axes[0, 0].get_legend()
    leg_bottom = axes[1, 0].get_legend()

    # remove them from axes
    axes[0, 0].legend_.remove()
    axes[1, 0].legend_.remove()

    # IMPORTANT: anchor legends in FIGURE coordinates
    plt.tight_layout(rect=[0, 0, 0.88, 1])
    leg_top.set_bbox_to_anchor((0.88, 0.75), transform=fig.transFigure)
    leg_bottom.set_bbox_to_anchor((0.88, 0.25), transform=fig.transFigure)
    leg_top.set_loc("center left")
    leg_bottom.set_loc("center left")

    # add them back to the figure
    fig.add_artist(leg_top)
    fig.add_artist(leg_bottom)

    # plt.show()

    # plt.title(f"Noise level: {noise_level}, Dropout level: {dropout_level}")
    if save_path is not None:
        plt.savefig(f"{save_path}/all_methods_embeddings.png", bbox_inches='tight', format="png")
        plt.savefig(f"{save_path}/all_methods_embeddings.pdf", bbox_inches='tight', format="pdf")

In [ ]:
# plot_embeddings(methods = ["Unintegrated", "FoSTA_PHATE_t2", "scANVI"], noise_level=0.5, dropout_level=0.5)

In [ ]:
import phate
import rfphate

# def plot_batches(methods = None, noise_level=0.5, dropout_level=0.5):
def add_rfphate_phate_embeddings(adata = None, label_key = "cell_type", batch_key = "simulated_batch", noise_level=0.5, dropout_level=0.5, embedding_basis = "X_pca", t=2):
    if adata is None:
        adata, path = load_adata(noise_level=noise_level, dropout_level=dropout_level)
    else:
        path = None
    phate_operator = phate.PHATE(n_components=2, t=t)
    rfphate_operator = rfphate.RFPHATE(n_components=2, t=t)

    phate_embeddings = phate_operator.fit_transform(adata.obsm[embedding_basis])
    rfphate_embeddings = rfphate_operator.fit_transform(adata.obsm[embedding_basis], adata.obs[label_key])

    adata.obsm[f"PHATE_t{t}"] = phate_embeddings
    adata.obsm[f"RFPHATE_t{t}"] = rfphate_embeddings
    
    return adata, path

In [ ]:
ts = [2, 4, 8, 12]
# ks = [5, 10, 20]
k=0
# for k in ks:
for t in ts:
    adata_no_noise, path = add_rfphate_phate_embeddings(adata=None, noise_level=0.0, dropout_level=0.0, t=t, knn =k)
    adata_t, path = add_rfphate_phate_embeddings(adata=None, noise_level=0.5, dropout_level=0.5, t=t, knn=k)
    
    # adatae = adata_t.obsm[f"PHATE_{t}_{k}"]
    # emb_rfphate = adata_t.obsm[f"RFPHATE_{t}_{k}"]
    # adata_t.obsm[f"PHATE_no_noise_{t}_{k}"] = adata_no_noise.obsm[f"PHATE_{t}_{k}"]
    adata_t.obsm[f"RFPHATE_no_noise_{t}_{k}"] = adata_no_noise.obsm[f"RFPHATE_{t}_{k}"]
    # plot_embeddings(adata_t, methods = [f"PHATE_{t}_{k}", f"RFPHATE_{t}_{k}",  f"RFPHATE_no_noise_{t}_{k}" , f"PHATE_no_noise_{t}_{k}"])
    # plot_embeddings(adata_t, methods = [f"PHATE_{t}_{k}", f"PHATE_no_noise_{t}_{k}"])
    plot_embeddings(adata_t, methods = [f"RFPHATE_{t}", f"RFPHATE_no_noise_{t}"])
    plt.show()

good plot

In [ ]:
adata_no_noise, path = add_rfphate_phate_embeddings(adata=None, noise_level=0.0, dropout_level=0.0, t=2)
adata, path = add_rfphate_phate_embeddings(adata=None, noise_level=0.5, dropout_level=0.5, t=2)

adata.obsm["PHATE_no_noise_t2"] = adata_no_noise.obsm["PHATE_t2"]
adata.obsm["RFPHATE_no_noise_t2"] = adata_no_noise.obsm["RFPHATE_t2"]

In [ ]:
# rename RFPHATE_t2 to RFPHATE
adata.obsm["RFPHATE"] = adata.obsm["RFPHATE_t2"]
# del adata.obsm["RFPHATE_t2"]


renames = {"PHATE_no_noise_t2": "PHATE: Before batch effects",
           "RFPHATE_no_noise_t2": "Before batch effects",
            "PHATE_t2": "Uncorrected batch effects",
            "RFPHATE" : "Uncorrected batch effects",
            "FoSTA": "FoSTA: Corrected batch effects"}

# methods = ["PHATE_no_noise_t2", 
# methods = ["RFPHATE_no_noise_t2", 
        #    "PHATE_t2", 
methods = ["RFPHATE", 
           "FoSTA"]

plot_embeddings(adata, methods = methods, save_path = path, rename_dict=renames, figsize=(8,6), s=30)

In [ ]:
adata.obs["cell_type"].cat.categories

In [ ]:
adata.write_h5ad(f"{path}/adata_with_phate_rfphate.h5ad")

# clean up cell types

In [ ]:
adata.obs["original_cell_type"] = adata.obs["cell_type"]

threshold = 10

value_counts = adata.obs["cell_type"].value_counts()
small_types = value_counts.index[value_counts < threshold]

# replace cell_type "Type 2" with "Alveolar Type 2" 
adata.obs.replace({"cell_type": {"Type 2": "Alveolar Type 2"}}, inplace=True)


for small_type in small_types:
    adata.obs.replace({"cell_type": {small_type: "Other"}}, inplace=True)



In [ ]:


# reorder the categories so that "Other" is last
categories = [cat for cat in adata.obs["cell_type"].cat.categories if cat != "Other"]
categories.append("Other")
adata.obs["cell_type"] = adata.obs["cell_type"].cat.reorder_categories(categories)

In [ ]:
# rerun the plot

plot_embeddings(adata, methods = methods, save_path = path, rename_dict=renames, figsize=(8,6), s=30)